# Inpainting demo-molecule selection

Rank a shortlist of candidate products for the inpainting demo based on:
1. How diverse `predict_precursors` output is (more distinct = more room to inpaint to).
2. Whether `predict_with_inpainting` actually returns alternatives under locking different sub-regions.

Winner gets wired as the default SMILES in `wsgi.py`.

In [ ]:
import sys, time, warnings
sys.path.insert(0, '/home/laabidn1/laabidn1/RetroLab/DiffAlign')
warnings.filterwarnings('ignore')

from rdkit import RDLogger; RDLogger.DisableLog('rdApp.*')
from rdkit import Chem
from rdkit.Chem import AllChem, DataStructs

from diffalign.inference import (
    predict_precursors,
    predict_with_inpainting,
    get_model_and_cfg,
)

_ = get_model_and_cfg()
print('model loaded')

## Shortlist

In [ ]:
# Candidates pre-screened from USPTO-50k eval file (>=3 distinct predicted
# precursor sets, 15-25 heavy atoms) plus iconic drug molecules as fallback.
CANDIDATES = [
    ('aspirin',         'CC(=O)Oc1ccccc1C(=O)O'),
    ('caffeine',        'CN1C=NC2=C1C(=O)N(C(=O)N2C)C'),
    ('ibuprofen',       'CC(C)Cc1ccc(cc1)C(C)C(=O)O'),
    # Top USPTO-50k candidates (ranked by #distinct predicted precursor sets)
    ('uspto_pyridyl_aryl_ester',  'CCOC1=CC(C2=CC=CC=N2)=C(C2CC2)C=C1C(=O)OC'),   # 22 atoms, 68 distinct
    ('uspto_chlorobipyridine',    'N#CC1=CC=CN=C1C1=CC=CC(Cl)=N1'),              # 15 atoms, 67 distinct
    ('uspto_phenylacetamide',     'O=C(CC1=CC=CC=C1)N1CCC2=C(C=NC3=C2C=NN3)C1'), # 22 atoms, 64 distinct
]

def n_heavy(smi):
    m = Chem.MolFromSmiles(smi)
    return m.GetNumHeavyAtoms() if m else -1

for name, smi in CANDIDATES:
    print(f'{name:32s} atoms={n_heavy(smi):2d}  {smi}')

## Pass 1: rank by diversity of `predict_precursors`

In [ ]:
def tanimoto_matrix(smiles_list):
    mols = [Chem.MolFromSmiles(s) for s in smiles_list]
    fps = [AllChem.GetMorganFingerprintAsBitVect(m, 2, 1024) for m in mols if m]
    n = len(fps)
    if n < 2: return 0.0
    sims = []
    for i in range(n):
        for j in range(i+1, n):
            sims.append(DataStructs.TanimotoSimilarity(fps[i], fps[j]))
    return sum(sims) / len(sims) if sims else 0.0

PASS1_N, PASS1_STEPS = 20, 20
pass1 = []
for name, smi in CANDIDATES:
    t = time.time()
    res = predict_precursors(product_smiles=smi, n_precursors=PASS1_N, diffusion_steps=PASS1_STEPS)
    elapsed = time.time() - t
    n_distinct = len(res)
    top_score = res[0]['score'] if res else 0.0
    top3 = [r['precursors'] for r in res[:3]]
    mean_tanimoto = tanimoto_matrix(top3)  # lower = more diverse
    pass1.append({
        'name': name, 'smi': smi, 'elapsed': elapsed,
        'n_distinct': n_distinct, 'top_score': top_score,
        'mean_tanimoto_top3': mean_tanimoto, 'results': res,
    })
    print(f'{name:32s} t={elapsed:4.1f}s  distinct={n_distinct:2d}  top={top_score:.2f}  T_top3={mean_tanimoto:.2f}')

In [ ]:
# Diversity score: more distinct precursors and lower Tanimoto similarity both help.
# Skip candidates with only 1 distinct (nothing to inpaint towards).
def rank_key(row):
    # more distinct is better; lower tanimoto is better (1.0 = identical)
    return (-row['n_distinct'], row['mean_tanimoto_top3'])

ranked = sorted([r for r in pass1 if r['n_distinct'] >= 2], key=rank_key)
print('Pass 1 ranking (most diverse first):')
for r in ranked:
    print(f"  {r['name']:32s}  distinct={r['n_distinct']:2d}  T_top3={r['mean_tanimoto_top3']:.2f}  {r['smi']}")

## Pass 2: inpaint top 2 candidates

For each top candidate: take its best precursor, lock two different sub-regions, and confirm `predict_with_inpainting` returns new alternatives (not just the same set) without tripping the stuck-atoms filter.

In [ ]:
PASS2_N, PASS2_STEPS = 20, 20

def total_real_nodes(atom_mapping):
    nodes = set()
    for mi in atom_mapping:
        nodes.update(mi['atom_map'].values())
    return nodes

def inpaint_trial(name, smi, result, lock_label, lock_fn):
    """Run inpainting with `lock_fn(atom_mapping) -> list[int]` to choose nodes to keep."""
    all_nodes = total_real_nodes(result['atom_mapping'])
    lock_nodes = lock_fn(result['atom_mapping'])
    if not lock_nodes or len(lock_nodes) >= len(all_nodes):
        print(f'  [{lock_label}] skipped (lock empty or full)')
        return
    t = time.time()
    new_res, failure = predict_with_inpainting(
        product_smiles=smi,
        previous_sample_data=result['sample_data'],
        inpaint_node_indices=list(lock_nodes),
        n_precursors=PASS2_N, diffusion_steps=PASS2_STEPS,
    )
    dt = time.time() - t
    if failure:
        print(f'  [{lock_label}] lock={len(lock_nodes)}/{len(all_nodes)} t={dt:.1f}s  FAILED: stuck={len(failure["stuck_atoms"])}')
        return
    # Compare to original top
    original = result['precursors']
    new_prec = [r['precursors'] for r in new_res]
    different = [p for p in new_prec if p != original]
    print(f'  [{lock_label}] lock={len(lock_nodes)}/{len(all_nodes)} t={dt:.1f}s  got={len(new_res)} different={len(different)}')
    for p in new_prec[:3]:
        flag = '=' if p == original else '*'
        print(f'     {flag} {p}')

def lock_first_half(atom_mapping):
    # Lock the first molecule entirely (change only the second)
    if len(atom_mapping) == 0: return []
    return list(atom_mapping[0]['atom_map'].values())

def lock_second_half(atom_mapping):
    if len(atom_mapping) < 2: return []
    nodes = []
    for mi in atom_mapping[1:]:
        nodes.extend(mi['atom_map'].values())
    return nodes

def lock_all_but_three(atom_mapping):
    all_nodes = sorted(total_real_nodes(atom_mapping))
    return all_nodes[:-3] if len(all_nodes) > 3 else []

for r in ranked[:2]:
    if not r['results']:
        continue
    print(f'\n=== {r["name"]} ({r["smi"]}) ===')
    best = r['results'][0]
    print(f'  base precursors: {best["precursors"]}')
    inpaint_trial(r['name'], r['smi'], best, 'lock-first-mol', lock_first_half)
    inpaint_trial(r['name'], r['smi'], best, 'lock-other-mols', lock_second_half)
    inpaint_trial(r['name'], r['smi'], best, 'lock-all-but-3', lock_all_but_three)

## Winner

In [ ]:
# Print final summary sorted by: (#distinct, base Tanimoto diversity).
print('FINAL SUMMARY')
print('-' * 80)
for r in ranked:
    print(f'{r["name"]:32s} distinct={r["n_distinct"]:2d}  T_top3={r["mean_tanimoto_top3"]:.2f}')
    print(f'  smi: {r["smi"]}')
    if r['results']:
        print(f'  top: {r["results"][0]["precursors"]}')
print('-' * 80)
winner = ranked[0] if ranked else None
if winner:
    print(f'\nWINNER: {winner["name"]}\n  SMILES: {winner["smi"]}')